In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [2]:
sentence = "the cat sits on the mat"

words = sentence.split()
vocab = sorted(set(words))

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

print("Words:", words)
print("Vocabulary:", word_to_id)

Words: ['the', 'cat', 'sits', 'on', 'the', 'mat']
Vocabulary: {'cat': 0, 'mat': 1, 'on': 2, 'sits': 3, 'the': 4}


In [3]:
X = []
y = []

for i in range(1, len(words) - 1):
    context = [
        word_to_id[words[i - 1]],
        word_to_id[words[i + 1]]
    ]

    target = word_to_id[words[i]]

    X.append(context)
    y.append(target)

X = np.array(X)
y = np.array(y)

print("Training contexts (X):")
print(X)
print("Target word IDs (y):")
print(y)
print("Example context words:", [id_to_word[i] for i in X[1]])
print("Example target word:", id_to_word[y[1]])

Training contexts (X):
[[4 3]
 [0 2]
 [3 4]
 [2 1]]
Target word IDs (y):
[0 3 2 4]
Example context words: ['cat', 'on']
Example target word: sits


In [4]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Embedding(len(vocab), 10),
    layers.GlobalAveragePooling1D(),
    layers.Dense(len(vocab), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 2, 10)          │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 10)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │            55 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105 (420.00 B)

 Trainable params: 105 (420.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(X, y, epochs=200, verbose=0)

print("Training completed.")
print("Final training accuracy:", round(float(history.history["accuracy"][-1]), 3))
print("Final training loss:", round(float(history.history["loss"][-1]), 4))

Training completed.
Final training accuracy: 0.75
Final training loss: 1.0654


In [6]:
context_words = ["cat", "on"]

context_ids = np.array([[
    word_to_id[context_words[0]],
    word_to_id[context_words[1]]
]])

prediction = model.predict(context_ids, verbose=0)

predicted_id = np.argmax(prediction)
predicted_word = id_to_word[predicted_id]

print("Context words:", context_words)
print("Predicted word:", predicted_word)
print("Probability:", round(float(prediction[0][predicted_id]), 3))

Context words: ['cat', 'on']
Predicted word: sits
Probability: 0.425
